## Notebook 07 — Combined Cross-Dataset Evaluation

Uses the **whuGAIT-trained** CNN + authenticator and evaluates on combined
ucihar + wisdm test subjects (all non-members relative to whuGAIT training).

Sections:
1. Load whuGAIT model and combined test data
2. Authentication evaluation per dataset
3. MIA simple-delta signal per dataset
4. LiRA shadow models (K=16, trained on whuGAIT members)
5. Evasion attack subset (sampled pairs per dataset)

In [ ]:
import sys, json, time, logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '..')

import torch
from sklearn.metrics import roc_curve, auc as sk_auc

from src.data.auth_dataset   import normalize_auth
from src.data.pairs_loader   import (load_auth_pairs, load_attribution,
                                     build_pair_subjects_train, build_pair_subjects_test)
from src.models.gait_cnn     import GaitCNN
from src.models.auth_model   import AuthModel
from src.attacks.lira_shadow import (train_shadow_models, all_deltas,
                                     compute_subject_scores, lira_eval, tpr_at_fpr)
from src.attacks.pgd         import run_pgd_sensor_batched, batch_psame, attack_success_rate, psr
from src.utils.config_loader import dataset_files
from src.utils.latex_writer  import write_latex_metrics

# ── Config ────────────────────────────────────────────────────────────────────
DATASETS_TEST   = ['ucihar', 'wisdm']
OFFSETS         = {'ucihar': 1000, 'wisdm': 2000}
DATA_ROOT       = Path('../data')
K_SHADOW        = 16
SHADOW_EPOCHS   = 3
N_PAIRS_EVASION = 500      # pairs per dataset for evasion
SEED            = 42
K_PGD           = 40
BATCH           = 512
DROPOUT         = 0.3
DEVICE          = torch.device('cpu')

LOG_DIR_WHU  = Path('../logs')     / 'whuGAIT'
CKPT_DIR_WHU = Path('../checkpoints') / 'whuGAIT'
LOG_DIR_OUT  = Path('../logs')     / 'combined'
RESULT_DIR   = Path('../results')  / 'combined'
for d in [LOG_DIR_OUT, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

log = logging.getLogger('nb07')
log.setLevel(logging.DEBUG)
log.handlers.clear()
fh = logging.FileHandler(LOG_DIR_OUT / '07_combined_eval.log', mode='w')
fh.setFormatter(logging.Formatter('%(asctime)s  %(message)s', datefmt='%H:%M:%S'))
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter('%(message)s'))
log.addHandler(fh); log.addHandler(sh)
log.info('=== Notebook 07 — Combined Cross-Dataset Evaluation [whuGAIT → ucihar + wisdm] ===')

In [ ]:
paths = dataset_files(LOG_DIR_WHU, CKPT_DIR_WHU)

n_classes = json.load(open(paths['cnn_meta']))['n_classes']
cnn = GaitCNN(n_classes=n_classes)
cnn.load_state_dict(torch.load(paths['cnn_ckpt'], map_location='cpu'))
cnn.eval()

auth_model = AuthModel(cnn, dropout=DROPOUT).to(DEVICE)
auth_model.load_state_dict(torch.load(paths['auth_ckpt'], map_location='cpu'))
auth_model.eval()

nd           = np.load(paths['norm_stats'])
norm_mean    = nd['mean']
norm_std     = nd['std']

with open(paths['split_json']) as f:
    split = json.load(f)
member_ids   = split['train_ids']
member_set   = set(member_ids)
member_list  = sorted(member_ids)

# Load whuGAIT NB06a setup for eps_target and EPS_GRID
setup_06a = np.load(LOG_DIR_WHU / '06a_whuGAIT_attack_setup.npz', allow_pickle=False)
EPS_GRID   = setup_06a['eps_grid'].astype(np.float64)
eps_target = float(setup_06a['eps_target'])
N_BISECT   = int(setup_06a['n_bisect'])

log.info(f'whuGAIT CNN n_classes={n_classes}  members={len(member_list)}')
log.info(f'eps_target={eps_target:.3f}  EPS_GRID={EPS_GRID}')
print(f'CNN loaded (n_classes={n_classes})  members={len(member_list)}')
print(f'eps_target={eps_target:.3f}  grid={len(EPS_GRID)} points  bisect={N_BISECT}')

In [ ]:
def extract_feats(cnn, X1_n, X2_n, batch_size=512):
    """Concatenated CNN feature maps (B, 32, 128) for shadow LSTM input."""
    parts = []
    with torch.no_grad():
        for s in range(0, len(X1_n), batch_size):
            e = s + batch_size
            f1 = cnn.get_feature_maps(torch.from_numpy(X1_n[s:e]).float())
            f2 = cnn.get_feature_maps(torch.from_numpy(X2_n[s:e]).float())
            parts.append(torch.cat([f1, f2], dim=1).half().cpu())
    return torch.cat(parts, dim=0)

# ── whuGAIT training pairs (for shadow model training) ────────────────────────
log.info('Loading whuGAIT training pairs...')
X1_whu_tr, X2_whu_tr, y_whu_tr = load_auth_pairs('train', DATA_ROOT, LOG_DIR_WHU)
X1_whu_n,  X2_whu_n,  _        = normalize_auth(X1_whu_tr, X2_whu_tr, (norm_mean, norm_std))

sw1_whu_tr, sw2_whu_tr = load_attribution('train', LOG_DIR_WHU)
pair_subj_whu_tr = build_pair_subjects_train(
    y_whu_tr, sw1_whu_tr, sw2_whu_tr, member_set, logs_dir=LOG_DIR_WHU)

log.info('Extracting whuGAIT training CNN features...')
tr_feats = extract_feats(cnn, X1_whu_n, X2_whu_n)
log.info(f'  tr_feats: {tuple(tr_feats.shape)}')
print(f'whuGAIT train: {len(y_whu_tr)} pairs  feats={tuple(tr_feats.shape)}')

# ── Cross-dataset test pairs ───────────────────────────────────────────────────
ds_data = {}   # ds → dict with X1_n, X2_n, y, subj1, subj2, feats
for ds in DATASETS_TEST:
    log_dir_ds = Path('../logs') / ds
    if not log_dir_ds.exists():
        log.warning(f'{ds}: logs dir not found — skipping')
        print(f'  SKIP {ds} (not found)')
        continue
    log.info(f'Loading {ds} test pairs...')
    X1, X2, y = load_auth_pairs('test', DATA_ROOT, log_dir_ds)
    X1_n, X2_n, _ = normalize_auth(X1, X2, (norm_mean, norm_std))  # whuGAIT norm stats
    sw1, sw2 = load_attribution('test', log_dir_ds)
    offset = OFFSETS[ds]
    sw1_off = np.where(sw1 >= 0, sw1 + offset, sw1)
    sw2_off = np.where(sw2 >= 0, sw2 + offset, sw2)
    log.info(f'Extracting {ds} CNN features...')
    te_feats_ds = extract_feats(cnn, X1_n, X2_n)
    ds_data[ds] = dict(X1_n=X1_n, X2_n=X2_n, y=y,
                       sw1=sw1_off, sw2=sw2_off, feats=te_feats_ds)
    n_subj = len(set(sw1_off[sw1_off >= 0]) | set(sw2_off[sw2_off >= 0]))
    log.info(f'  {ds}: {len(y)} pairs  subjects≈{n_subj}  feats={tuple(te_feats_ds.shape)}')
    print(f'  {ds}: {len(y)} pairs  ~{n_subj} subjects  feats={tuple(te_feats_ds.shape)}')

# Combined test arrays
te_X1_n  = np.concatenate([ds_data[ds]['X1_n'] for ds in ds_data])
te_X2_n  = np.concatenate([ds_data[ds]['X2_n'] for ds in ds_data])
te_y     = np.concatenate([ds_data[ds]['y']    for ds in ds_data])
te_sw1   = np.concatenate([ds_data[ds]['sw1']  for ds in ds_data])
te_sw2   = np.concatenate([ds_data[ds]['sw2']  for ds in ds_data])
te_feats = torch.cat([ds_data[ds]['feats'] for ds in ds_data], dim=0)

nonmember_ids  = sorted(set(te_sw1[te_sw1 >= 0]) | set(te_sw2[te_sw2 >= 0]))
nonmember_set  = set(nonmember_ids)

# te_pair_subjects: non-member subject per test pair
te_pair_subj = build_pair_subjects_test(
    te_y, te_sw1, te_sw2, nonmember_set)

log.info(f'Combined test: {len(te_y)} pairs  {len(nonmember_ids)} non-member subjects')
print(f'Combined test: {len(te_y)} pairs  {len(nonmember_ids)} non-member subjects  '
      f'feats={tuple(te_feats.shape)}')

In [ ]:
log.info('Computing auth scores...')

def auth_scores_batched(model, X1_n, X2_n, batch=512):
    scores = []
    with torch.no_grad():
        for s in range(0, len(X1_n), batch):
            x1 = torch.from_numpy(X1_n[s:s+batch]).float()
            x2 = torch.from_numpy(X2_n[s:s+batch]).float()
            scores.append(batch_psame(model, x1, x2, batch_size=batch))
    return np.concatenate(scores)

fig, axes = plt.subplots(1, len(ds_data)+1, figsize=(5*(len(ds_data)+1), 4))
if len(ds_data) == 1:
    axes = [axes, axes]

results_auth = {}
for ax, ds in zip(axes, list(ds_data) + ['combined']):
    if ds == 'combined':
        X1_plot, X2_plot, y_plot = te_X1_n, te_X2_n, te_y
        title = 'Combined'
    else:
        X1_plot = ds_data[ds]['X1_n']
        X2_plot = ds_data[ds]['X2_n']
        y_plot  = ds_data[ds]['y']
        title   = ds

    scores = auth_scores_batched(auth_model, X1_plot, X2_plot)
    fpr, tpr, _ = roc_curve(y_plot, scores)
    auc_val = sk_auc(fpr, tpr)
    acc = float(((scores > 0.5) == y_plot).mean())
    results_auth[ds] = dict(auc=auc_val, acc=acc)

    ax.plot(fpr, tpr, lw=1.5, label=f'AUC={auc_val:.4f}')
    ax.plot([0,1],[0,1],'--',color='gray',lw=0.8)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'Auth ROC — {title}')
    ax.legend(); ax.grid(True, alpha=0.3)
    log.info(f'Auth {ds}: AUC={auc_val:.4f}  acc={acc:.4f}')
    print(f'  {title:<12} AUC={auc_val:.4f}  acc={acc*100:.1f}%')

plt.tight_layout()
plt.savefig(RESULT_DIR / '07_auth_roc.png', dpi=150)
plt.show()
log.info('Auth evaluation done')

In [ ]:
log.info('Computing MIA simple delta...')

# Load whuGAIT NB05a results for reference member/non-member distributions
d05a = np.load(LOG_DIR_WHU / '05a_target_deltas.npz', allow_pickle=True)
whu_member_deltas    = d05a['member_deltas']
whu_nonmember_deltas = d05a['nonmember_deltas']

def per_subject_delta(model, X1_n, X2_n, y, sw1, sw2, batch=512):
    """Simple delta per subject: mean(same_score) - mean(diff_score)."""
    scores = auth_scores_batched(model, X1_n, X2_n, batch)
    subj_ids = set(sw1[sw1 >= 0]) | set(sw2[sw2 >= 0])
    deltas, sids = [], []
    for s in sorted(subj_ids):
        mask1 = (sw1 == s) | (sw2 == s)
        if mask1.sum() < 5:
            continue
        same_sc = scores[mask1 & (y == 0)]
        diff_sc = scores[mask1 & (y == 1)]
        if len(same_sc) == 0 or len(diff_sc) == 0:
            continue
        raw, _, _ = all_deltas(same_sc, diff_sc)
        if raw is not None:
            deltas.append(raw); sids.append(s)
    return np.array(deltas), np.array(sids)

results_mia = {}
all_cross_deltas = []
for ds in ds_data:
    d = ds_data[ds]
    deltas, sids = per_subject_delta(
        auth_model, d['X1_n'], d['X2_n'], d['y'], d['sw1'], d['sw2'])
    results_mia[ds] = dict(deltas=deltas, sids=sids)
    all_cross_deltas.append(deltas)
    log.info(f'Delta {ds}: n={len(deltas)}  mean={deltas.mean():.4f}  std={deltas.std():.4f}')
    print(f'  {ds}: {len(deltas)} subjects  mean_delta={deltas.mean():.4f}')

all_cross_deltas = np.concatenate(all_cross_deltas)

# AUC: whuGAIT members (positive) vs cross-dataset non-members (negative)
from sklearn.metrics import roc_curve, auc as sk_auc
combined_deltas = np.concatenate([whu_member_deltas, all_cross_deltas])
combined_labels = np.array([1]*len(whu_member_deltas) + [0]*len(all_cross_deltas))
fpr_d, tpr_d, _ = roc_curve(combined_labels, combined_deltas)
auc_delta = sk_auc(fpr_d, tpr_d)
tpr10_delta = tpr_at_fpr(fpr_d, tpr_d, 0.10)

log.info(f'MIA delta  whuGAIT-members vs cross-dataset: AUC={auc_delta:.4f}  TPR@10%={tpr10_delta:.3f}')
print(f'MIA delta AUC (whuGAIT-members vs cross-dataset non-members): {auc_delta:.4f}  TPR@10%={tpr10_delta:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.hist(whu_member_deltas,    bins=30, alpha=0.6, label=f'whuGAIT members (n={len(whu_member_deltas)})', color='#2ecc71')
for ds in results_mia:
    ax.hist(results_mia[ds]['deltas'], bins=20, alpha=0.6, label=f'{ds} non-members (n={len(results_mia[ds]["deltas"])})')
ax.set_xlabel('Delta (simple)'); ax.set_ylabel('Count')
ax.set_title('Delta distribution: members vs cross-dataset')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(fpr_d, tpr_d, lw=1.5, label=f'AUC={auc_delta:.4f}  TPR@10%={tpr10_delta:.3f}')
ax.plot([0,1],[0,1],'--',color='gray',lw=0.8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('MIA ROC — members vs cross-dataset non-members')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '07_mia_delta_roc.png', dpi=150)
plt.show()

In [ ]:
log.info(f'Training {K_SHADOW} shadow LSTMs on whuGAIT training data...')
print(f'Training {K_SHADOW} shadow LSTMs (init=target, epochs={SHADOW_EPOCHS})...')

target_state = {
    'lstm': auth_model.lstm.state_dict(),
    'fc':   auth_model.fc.state_dict(),
}

CKPT_SHADOW = LOG_DIR_OUT / '07_combined_shadow_ckpt.npz'

out_raw, out_lf, out_mf = train_shadow_models(
    tr_feats, y_whu_tr, pair_subj_whu_tr, member_list,
    te_feats, te_y,     te_pair_subj,     nonmember_ids,
    K=K_SHADOW, epochs=SHADOW_EPOCHS,
    init_mode='target', target_state=target_state,
    split_mode='member_aware',
    pair_subjects_w2=sw2_whu_tr,
    checkpoint_path=CKPT_SHADOW,
    batch_size=BATCH, device=str(DEVICE), log=log,
    dropout=DROPOUT,
)

valid_nm = [s for s in nonmember_ids if s in out_raw and len(out_raw[s]) >= 5]
log.info(f'Valid non-members (≥5 OUT estimates): {len(valid_nm)}/{len(nonmember_ids)}')
print(f'Valid non-members: {len(valid_nm)}/{len(nonmember_ids)}')

# LiRA scores: non-members only (all test subjects are non-members)
# Compare to whuGAIT member deltas from NB05b
d05b = json.load(open(LOG_DIR_WHU / '05b_per_subject_report.json'))
whu_member_lira_raw = np.array([d05b[str(s)]['lira_raw'] for s in d05b if d05b[str(s)].get('lira_raw') is not None])

cross_lira_raw = np.array([np.mean(out_raw[s]) for s in valid_nm])

combined_lira  = np.concatenate([whu_member_lira_raw, cross_lira_raw])
combined_lira_y = np.array([1]*len(whu_member_lira_raw) + [0]*len(cross_lira_raw))
fpr_l, tpr_l, _ = roc_curve(combined_lira_y, combined_lira)
auc_lira = sk_auc(fpr_l, tpr_l)
tpr10_lira = tpr_at_fpr(fpr_l, tpr_l, 0.10)

log.info(f'LiRA cross-dataset: AUC={auc_lira:.4f}  TPR@10%={tpr10_lira:.3f}')
print(f'LiRA cross-dataset AUC={auc_lira:.4f}  TPR@10%={tpr10_lira:.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(fpr_l, tpr_l, lw=1.5, label=f'LiRA cross-dataset AUC={auc_lira:.4f}')
ax.plot([0,1],[0,1],'--',color='gray',lw=0.8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('LiRA ROC — whuGAIT members vs cross-dataset non-members')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / '07_lira_roc.png', dpi=150)
plt.show()

In [ ]:
log.info('Running evasion attack subset...')
rng = np.random.default_rng(SEED)

results_evasion = {}
for ds in ds_data:
    d = ds_data[ds]
    diff_mask = d['y'] == 1
    idx_diff  = np.where(diff_mask)[0]
    n_sample  = min(N_PAIRS_EVASION, len(idx_diff))
    chosen    = rng.choice(idx_diff, n_sample, replace=False)

    X1_ev = torch.from_numpy(d['X1_n'][chosen]).float()
    X2_ev = torch.from_numpy(d['X2_n'][chosen]).float()

    scores_clean = batch_psame(auth_model, X1_ev, X2_ev, batch_size=BATCH)
    baseline_asr = attack_success_rate(scores_clean, target_label=1)

    log.info(f'{ds} evasion: {n_sample} pairs  eps_target={eps_target:.3f}  baseline_ASR={baseline_asr:.3f}')
    print(f'{ds}: attacking {n_sample} pairs at eps_target={eps_target:.3f}  baseline_ASR={baseline_asr:.3f}')

    X1_adv = run_pgd_sensor_batched(auth_model, X1_ev, X2_ev,
                                     target_label=0, eps=eps_target, K=K_PGD, batch_size=256)
    scores_adv = batch_psame(auth_model, X1_adv, X2_ev, batch_size=BATCH)
    asr   = attack_success_rate(scores_adv, target_label=1)
    psr_v = psr(X1_ev, X1_adv)
    mean_l2 = float((X1_adv - X1_ev).reshape(len(X1_ev), -1).norm(dim=1).mean().item())

    results_evasion[ds] = dict(asr=asr, psr=psr_v, mean_l2=mean_l2,
                                baseline_asr=baseline_asr, n_pairs=n_sample)
    log.info(f'{ds}: ASR={asr:.3f}  PSR={psr_v:.4f}  mean_l2={mean_l2:.4f}')
    print(f'  ASR={asr:.3f}  PSR={psr_v:.4f} ({psr_v*100:.1f}%)  mean_l2={mean_l2:.4f}')

log.info('Evasion done')

In [ ]:
print()
print('='*60)
print('NOTEBOOK 07 SUMMARY — Cross-Dataset Evaluation')
print('='*60)
print(f'whuGAIT model: {n_classes} classes  eps_target={eps_target:.3f}')
print()
print('  Auth evaluation (whuGAIT model → cross-dataset test subjects):')
for ds, r in results_auth.items():
    print(f'    {ds:<12}  AUC={r["auc"]:.4f}  acc={r["acc"]*100:.1f}%')
print()
print('  MIA simple delta (whuGAIT members vs cross-dataset non-members):')
print(f'    AUC={auc_delta:.4f}  TPR@10%={tpr10_delta:.3f}')
for ds, r in results_mia.items():
    print(f'    {ds:<12}  n={len(r["deltas"])}  mean_delta={r["deltas"].mean():.4f}')
print()
print('  LiRA shadow (K={K_SHADOW} shadows on whuGAIT train):')
print(f'    AUC={auc_lira:.4f}  TPR@10%={tpr10_lira:.3f}')
print()
print('  Evasion attack subset ({N_PAIRS_EVASION} pairs/dataset):')
for ds, r in results_evasion.items():
    print(f'    {ds:<12}  ASR={r["asr"]:.3f}  PSR={r["psr"]*100:.1f}%  mean_l2={r["mean_l2"]:.3f}')

# ── LaTeX ─────────────────────────────────────────────────────────────────────
latex_metrics = {
    'auth_combined_auc':    results_auth.get('combined', {}).get('auc', float('nan')),
    'mia_cross_auc_delta':  auc_delta,
    'mia_cross_tpr10_delta': tpr10_delta,
    'mia_cross_auc_lira':   auc_lira,
    'mia_cross_tpr10_lira': tpr10_lira,
}
for ds, r in results_evasion.items():
    latex_metrics[f'evasion_{ds}_asr']  = r['asr']
    latex_metrics[f'evasion_{ds}_psr']  = r['psr']
for ds, r in results_auth.items():
    if ds != 'combined':
        latex_metrics[f'auth_{ds}_auc'] = r['auc']

write_latex_metrics(latex_metrics, tag='nb07_combined',
                    out_dir=Path('../latex/generated'))
print()
print('LaTeX metrics written: latex/generated/nb07_combined_metrics.tex')
log.info('=== NB07 complete ===')